on génère un problème de classification binaire où les variables explicatives sont indépendantes et uniformes entre 0 et 1.
La probabilité que Y=1 dépend de manière non-linéaire des deux premières variables, et on utilise la fonction de répartition normale pour transformer cette combinaison en une probabilité entre 0 et 1.
enfin, on génère la classe Y à partir d’une loi Bernoulli.
Ce scénario permet de tester des méthodes de classification dans un cadre simple mais non-linéaire.
X[:,0] = 𝑋 ( 1 )
X[:,1] = 𝑋 ( 2 ) 
norm.cdf() = fonction Φ (CDF normale standard)
Pourquoi utiliser une CDF normale ?
Pour obtenir des probabilités dans [0,1] à partir d’une combinaison linéaire.

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# Parameters
N = 100
p = 2  # at least 2 variables required

# Generate X ~ Unif[0,1]^p
X = np.random.uniform(0, 1, size=(N, p))

# Compute mu_i = Φ(10 * (X1 - 1) + 20 * |X2 - 0.5|)
mu = norm.cdf(10 * (X[:,0] - 1) + 20 * np.abs(X[:,1] - 0.5))

# Generate Y_i ~ Bernoulli(mu_i)
Y = np.random.binomial(1, mu)

# Build DataFrame
df = pd.DataFrame({
    "X1": X[:,0],
    "X2": X[:,1],
    "mu": mu,
    "Y": Y
})

# Save to CSV
df.to_csv("dataset_scenario1.csv", index=False)

print("Dataset saved as dataset_scenario1.csv")
print(df.head())


Dataset saved as dataset_scenario1.csv
         X1        X2        mu  Y
0  0.546326  0.680065  0.174778  0
1  0.649464  0.281126  0.808428  1
2  0.494024  0.789027  0.764474  1
3  0.765482  0.195823  0.999907  1
4  0.707528  0.487225  0.003801  0


on génère un modèle de régression non linéaire où les variables explicatives sont indépendantes et uniformes entre 0 et 1.
La réponse dépend uniquement des deux premières variables, à travers des termes tronqués (parties positives), ce qui crée une relation active seulement lorsque 𝑋 ( 1 ) > 0.5 X (1) >0.5 et 𝑋 ( 2 ) > 0.25 X (2) >0.25.
Cette structure introduit une non-linéarité forte, car la fonction s’annule pour une grande partie de l’espace des covariables, puis augmente rapidement ailleurs.
enfin, on ajoute un bruit gaussien pour simuler une réponse réelle.
np.maximum(a,0) = ( 𝑎 ) + 
(a) + = partie positive 
pourquoi la partie positive ? → Pour créer une fonction non linéaire qui s’active seulement au-delà d’un certain seuil.

In [6]:
import numpy as np
import pandas as pd

# Parameters
N = 100      # number of samples
p = 2        # dimensionality (we need at least 2)

# Generate X ~ Unif[0,1]^p
X = np.random.uniform(0, 1, size=(N, p))

# Noise
epsilon = np.random.normal(0, 1, size=N)

# Compute Y according to the model
# (x)^+ means max(x,0)
Y = 100 * np.maximum(X[:,0] - 0.5, 0)**2 * np.maximum(X[:,1] - 0.25, 0) + epsilon

# Build DataFrame
df = pd.DataFrame({
    "X1": X[:,0],
    "X2": X[:,1],
    "Y": Y
})

# Save to CSV
df.to_csv("dataset_scenario2.csv", index=False)

print("Dataset saved as dataset_scenario2.csv")
print(df.head())


Dataset saved as dataset_scenario2.csv
         X1        X2         Y
0  0.314185  0.054865  0.262101
1  0.544004  0.832631  0.126635
2  0.899708  0.477775  3.156501
3  0.146963  0.894922  0.414119
4  0.882637  0.208527 -0.724212


In [8]:
import numpy as np
import pandas as pd

# Parameters
N = 300
p = 200  # must be >= 200

# Build covariance matrix Σ_ij = 0.9^{|i-j|}
rho = 0.9
indices = np.arange(p)
Sigma = rho ** np.abs(indices.reshape(-1, 1) - indices.reshape(1, -1))

# Generate X ~ N(0, Σ)
X = np.random.multivariate_normal(mean=np.zeros(p), cov=Sigma, size=N)

# Noise
epsilon = np.random.normal(0, 1, size=N)

# Compute Y = 2 X50 X100 + 2 X150 X200 + noise
Y = (
    2 * X[:, 49] * X[:, 99] +   # X^(50) * X^(100)
    2 * X[:, 149] * X[:, 199] + # X^(150) * X^(200)
    epsilon
)

# Build DataFrame with X1,...,Xp
df = pd.DataFrame(X, columns=[f"X{i+1}" for i in range(p)])
df["Y"] = Y

# Save to CSV
df.to_csv("dataset_scenario3.csv", index=False)

print("Dataset saved as dataset_scenario3.csv")
print(df.head())


Dataset saved as dataset_scenario3.csv
         X1        X2        X3        X4        X5        X6        X7  \
0 -0.777406 -0.628644 -0.115078 -0.591915 -0.993137 -0.936888 -0.844087   
1 -0.040435  0.765411  0.776876  1.271700  1.935471  1.717900  1.692105   
2 -0.151986 -0.409350 -0.273594 -0.140309 -0.169247 -0.367110 -0.325508   
3  0.305080  0.119264  0.244427 -0.493604 -1.357532 -1.149817 -0.895673   
4 -0.981646 -0.900125 -0.915265 -0.199354  0.258285 -0.040494  0.521425   

         X8        X9       X10  ...      X192      X193      X194      X195  \
0 -0.101603  0.354370 -0.047575  ... -1.011889 -0.707488 -0.569868 -0.312377   
1  1.969920  1.515027  1.364574  ...  0.044886  0.602022  0.862274  0.790379   
2 -0.214971 -0.449778 -1.006609  ...  0.768117  0.935344  0.145344  0.362292   
3 -1.201568 -1.078850 -0.902731  ...  0.987914  1.246608  1.239058  1.347065   
4  0.892233  0.593498 -0.084094  ...  0.071683  0.944085  0.652911  0.030895   

       X196      X197      X1

In [10]:
import numpy as np
import pandas as pd

# Parameters
N = 200
p = 150  # must be >= 150

# Build covariance matrix Σ_ij = 0.5^{|i-j|} + 0.2 * I(i ≠ j)
indices = np.arange(p)
Sigma = 0.5 ** np.abs(indices.reshape(-1, 1) - indices.reshape(1, -1))
Sigma += 0.2 * (1 - np.eye(p))

# Generate X ~ N(0, Σ)
X = np.random.multivariate_normal(mean=np.zeros(p), cov=Sigma, size=N)

# Noise
epsilon = np.random.normal(0, 1, size=N)

# Compute Y = 2 X50 + 2 X100 + 4 X150 + noise
Y = 2*X[:, 49] + 2*X[:, 99] + 4*X[:, 149] + epsilon

# Build DataFrame with X1,...,Xp
df = pd.DataFrame(X, columns=[f"X{i+1}" for i in range(p)])
df["Y"] = Y

# Save to CSV
df.to_csv("dataset_scenario4.csv", index=False)

print("Dataset saved as dataset_scenario4.csv")
print(df.head())


Dataset saved as dataset_scenario4.csv
         X1        X2        X3        X4        X5        X6        X7  \
0  1.610430  0.098082 -0.620508  0.336223  0.597841 -0.064038  0.414333   
1  0.596356  0.926796  1.589658  0.606681  0.408159  1.561057  0.110977   
2 -1.181271 -1.807737 -0.716074 -0.981033 -0.315993  0.372565 -0.084562   
3  1.468797  0.651453  0.852803  0.703644  0.372698  0.894414  1.243642   
4 -1.992177 -2.302997 -1.376852  1.548988  1.026618 -0.528192 -0.975889   

         X8        X9       X10  ...      X142      X143      X144      X145  \
0  1.740453  1.779785  1.569848  ...  0.232164  0.737662  0.716581  1.005763   
1  0.221535  0.342919 -1.220315  ...  1.054666  0.081227 -0.343174 -0.784180   
2 -0.914100  0.851438  0.838003  ... -1.403119 -1.614535 -1.443652 -0.459910   
3  0.806045  0.978695  1.056165  ... -0.157872 -0.642826  1.065396  1.478737   
4 -1.404833 -0.866832 -0.508329  ... -0.613156 -0.327223  0.081907  0.625522   

       X146      X147      X1